In [12]:
import sys
import os
from pathlib import Path

# Add the parent directory to sys.path so 'dashboard' can be imported if it's a local package
current_dir = Path(os.getcwd())
parent_dir = current_dir.parent
if str(parent_dir) not in sys.path:
	sys.path.insert(0, str(parent_dir))

from dashboard.database.db_connection import DatabaseConnection
# Get database path

db_path = parent_dir / "dashboard" / "database" / "bikes.db"
print(db_path)
# Initialize database connection
db = DatabaseConnection(str(db_path))

INFO:dashboard.database.db_connection:Connected to database: /Users/ansen/Documents/GitHub/applied-data-science/dashboard/database/bikes.db


/Users/ansen/Documents/GitHub/applied-data-science/dashboard/database/bikes.db


In [8]:

sql = """
    SELECT date, hour, rented_bike_count, temperature, humidity, wind_speed
    FROM stg_bike_rentals_hourly
    WHERE date <= '2018-11-30'
    ORDER BY date, hour
    """
results = db.execute_query(sql)

INFO:dashboard.database.db_connection:Query executed successfully


In [9]:
print(results[:5])  # Print first 5 rows to verify data retrieval

[('2017-12-01', 0, 254, -5.2, 37, 2.2), ('2017-12-01', 1, 204, -5.5, 38, 0.8), ('2017-12-01', 2, 173, -6.0, 39, 1.0), ('2017-12-01', 3, 107, -6.2, 40, 0.9), ('2017-12-01', 4, 78, -6.0, 36, 2.3)]


In [13]:
import pandas as pd
from datetime import datetime, timedelta
# Parameters for data retrieval
selected_hour = 14
selected_date_str = "2018-11-23"

selected_date = datetime.strptime(selected_date_str, '%Y-%m-%d')
selected_datetime = datetime(selected_date.year, selected_date.month, selected_date.day, selected_hour, 0, 0)
week_ago = selected_datetime - timedelta(days=7)

# Query historical data (last 7 days)
hist_sql = f"""
        SELECT datetime(date || ' ' || printf('%02d:00:00', hour)) as datetime, 
               rented_bike_count as bikes
        FROM stg_bike_rentals_hourly
        WHERE datetime(date || ' ' || printf('%02d:00:00', hour)) >= '{week_ago}' 
        AND datetime(date || ' ' || printf('%02d:00:00', hour)) <= '{selected_datetime}'
        ORDER BY datetime
"""
hist_results = db.execute_query(hist_sql)
print(hist_results)

INFO:dashboard.database.db_connection:Query executed successfully


[('2018-11-16 14:00:00', 915), ('2018-11-16 15:00:00', 995), ('2018-11-16 16:00:00', 1126), ('2018-11-16 17:00:00', 1346), ('2018-11-16 18:00:00', 1956), ('2018-11-16 19:00:00', 1303), ('2018-11-16 20:00:00', 995), ('2018-11-16 21:00:00', 907), ('2018-11-16 22:00:00', 894), ('2018-11-16 23:00:00', 687), ('2018-11-17 00:00:00', 626), ('2018-11-17 01:00:00', 597), ('2018-11-17 02:00:00', 456), ('2018-11-17 03:00:00', 288), ('2018-11-17 04:00:00', 195), ('2018-11-17 05:00:00', 129), ('2018-11-17 06:00:00', 151), ('2018-11-17 07:00:00', 315), ('2018-11-17 08:00:00', 530), ('2018-11-17 09:00:00', 644), ('2018-11-17 10:00:00', 733), ('2018-11-17 11:00:00', 830), ('2018-11-17 12:00:00', 994), ('2018-11-17 13:00:00', 1037), ('2018-11-17 14:00:00', 969), ('2018-11-17 15:00:00', 1018), ('2018-11-17 16:00:00', 974), ('2018-11-17 17:00:00', 1031), ('2018-11-17 18:00:00', 877), ('2018-11-17 19:00:00', 735), ('2018-11-17 20:00:00', 644), ('2018-11-17 21:00:00', 685), ('2018-11-17 22:00:00', 655), ('

In [14]:

# Query 24-hour predictions (generated at selected time)
pred_24h_sql = f"""
SELECT prediction_datetime, predicted_bikes as prediction
FROM pred_bike_rentals_24h 
WHERE selected_date = '{selected_date_str}'
AND selected_hour = {selected_hour}
ORDER BY prediction_datetime
"""
pred_24h_results = db.execute_query(pred_24h_sql)

print(pred_24h_results)

INFO:dashboard.database.db_connection:Query executed successfully


[('2018-11-23 15:00:00', 766.3008229579366), ('2018-11-23 16:00:00', 840.6702557648822), ('2018-11-23 17:00:00', 1049.6102952921485), ('2018-11-23 18:00:00', 1522.1605648180719), ('2018-11-23 19:00:00', 1112.4365865972622), ('2018-11-23 20:00:00', 862.4858941709185), ('2018-11-23 21:00:00', 774.0203494129397), ('2018-11-23 22:00:00', 719.6394461964528), ('2018-11-23 23:00:00', 469.22927657852205), ('2018-11-24 00:00:00', 381.4804967728544), ('2018-11-24 01:00:00', 351.03069530235996), ('2018-11-24 02:00:00', 317.9105751339622), ('2018-11-24 03:00:00', 319.11730716494327), ('2018-11-24 04:00:00', 337.93873490397925), ('2018-11-24 05:00:00', 351.2035816471481), ('2018-11-24 06:00:00', 475.8794995500078), ('2018-11-24 07:00:00', 728.6201046286535), ('2018-11-24 08:00:00', 1137.4380774636472), ('2018-11-24 09:00:00', 715.475306008354), ('2018-11-24 10:00:00', 540.0978511650818), ('2018-11-24 11:00:00', 564.7603145128976), ('2018-11-24 12:00:00', 625.4567439589175), ('2018-11-24 13:00:00', 

In [5]:
pred_3d_sql = f"""
        SELECT prediction_datetime, predicted_bikes as prediction
        FROM pred_bike_rentals_3d 
        WHERE prediction_datetime = '2018-11-04 14:00:00'
        ORDER BY prediction_datetime
        """
pred_3d_results = db.execute_query(pred_3d_sql)
print(pred_3d_results)

INFO:dashboard.database.db_connection:Query executed successfully


[('2018-11-04 14:00:00', 812.989395275577)]


In [55]:
# Prepare DataFrames
hist_data = pd.DataFrame()
if hist_results:
    hist_data = pd.DataFrame(hist_results, columns=['datetime', 'bikes'])
    hist_data['datetime'] = pd.to_datetime(hist_data['datetime'])
    hist_data['bikes'] = pd.to_numeric(hist_data['bikes'], errors='coerce')

pred_24h_data = pd.DataFrame()
if pred_24h_results:
    pred_24h_data = pd.DataFrame(pred_24h_results, columns=['datetime', 'prediction'])
    pred_24h_data['datetime'] = pd.to_datetime(pred_24h_data['datetime'])
    pred_24h_data['prediction'] = pd.to_numeric(pred_24h_data['prediction'], errors='coerce')

pred_3d_data = pd.DataFrame()
if pred_3d_results:
    pred_3d_data = pd.DataFrame(pred_3d_results, columns=['datetime', 'prediction'])
    pred_3d_data['datetime'] = pd.to_datetime(pred_3d_data['datetime'])
    pred_3d_data['prediction'] = pd.to_numeric(pred_3d_data['prediction'], errors='coerce')

In [4]:
sql = f"""delete from pred_bike_rentals_3d"""
db.execute_query(sql)

sql = f"""delete from pred_bike_rentals_24h"""
db.execute_query(sql)

INFO:dashboard.database.db_connection:Query executed successfully
INFO:dashboard.database.db_connection:Query executed successfully


[]

In [15]:
db.commit()

In [16]:
db.close()

INFO:dashboard.database.db_connection:Database connection closed
